# 01 — Data and session repertoires

## Manuscript crosswalk

- **Methods:** Study subjects; Experimental design; Data processing; Analysis pipeline overview.
- **Results:** analysed-unit, eligibility, and session-comparison counts.
- **Figures:** Figure 1 (experimental design) and Figure S1 (analytical workflow).

This notebook establishes the units used by every later analysis. It reads processed tables and versioned session inventories, verifies their provenance, and checks the counts reported in the manuscript. It does not extract audio, segment calls, calculate distances, train a model, or fit a statistical model.

## Cached-only execution contract

The public notebooks never infer that a missing artifact should be rebuilt. All expensive switches below default to `False`. A missing file, missing manifest entry, checksum mismatch, or unexpected count stops execution with a specific error. Use the explicit full pipeline only when the complete source collection and suitable compute are available.

Canonical inputs:

- `data/processed/sequences.csv`
- `data/processed/calls.csv`
- `data/cache/sequence_session_inventory.csv`
- `data/cache/call_session_inventory.csv`
- `data/cache/sequence_session_pairs.csv`
- `data/cache/call_session_pairs.csv`

See [the data dictionary](../docs/data_dictionary.md), [analysis specification](../docs/analysis_specification.md), and [reproducibility guide](../docs/reproducibility.md).

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository without relying on a machine-specific absolute path."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter inside the cloned repository; "
        "the root must contain README.md and data/."
    )


ROOT = find_repo_root()
EXPENSIVE_STEPS = {
    "recompute_sequence": False,
    "recompute_dtw": False,
    "retrain_vae": False,
    "refit_models": False,
}
if any(EXPENSIVE_STEPS.values()):
    raise RuntimeError(
        "This reader-facing notebook is cached-only. Run "
        "'bash scripts/run_repro_pipeline.sh --mode full' explicitly instead."
    )

pd.DataFrame(
    {"setting": ["repository_root", *EXPENSIVE_STEPS],
     "value": [str(ROOT), *EXPENSIVE_STEPS.values()]}
)

In [ ]:
MANIFEST_PATHS = (
    ROOT / "data/processed/manifest.json",
    ROOT / "data/cache/manifest.json",
    ROOT / "data/derived/manifest.json",
    ROOT / "results/manifest.json",
)


def require_file(relative_path: str) -> Path:
    path = ROOT / relative_path
    if not path.is_file():
        raise FileNotFoundError(
            f"Required artifact is missing: {relative_path}. Cached mode will not "
            "recompute it. Obtain the complete release assets or run the relevant "
            "full-pipeline component explicitly."
        )
    return path


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_manifests() -> list[tuple[Path, dict]]:
    found = []
    for path in MANIFEST_PATHS:
        if path.is_file():
            with path.open(encoding="utf-8") as handle:
                found.append((path, json.load(handle)))
    if not found:
        raise FileNotFoundError(
            "No provenance manifest was found. Expected data/cache/manifest.json "
            "and, where applicable, derived/results manifests."
        )
    return found


def iter_file_records(payload: dict):
    for section_name in ("artifacts", "sources", "inputs", "files"):
        section = payload.get(section_name, {})
        if isinstance(section, dict):
            for key, value in section.items():
                record = value if isinstance(value, dict) else {"sha256": value}
                yield str(record.get("path", key)), record
        elif isinstance(section, list):
            for record in section:
                if isinstance(record, dict) and record.get("path"):
                    yield str(record["path"]), record


def checksum_from_record(record: dict) -> str | None:
    direct = record.get("sha256") or record.get("checksum_sha256")
    if direct:
        return str(direct).removeprefix("sha256:")
    checksum = record.get("checksum")
    if isinstance(checksum, str):
        return checksum.removeprefix("sha256:")
    if isinstance(checksum, dict) and checksum.get("algorithm", "").lower() == "sha256":
        return checksum.get("value")
    return None


MANIFESTS = load_manifests()


def require_registered(relative_path: str) -> Path:
    """Require a file and verify its registered SHA-256 before analysis."""
    path = require_file(relative_path)
    normalized = Path(relative_path).as_posix()
    matches = []
    for manifest_path, payload in MANIFESTS:
        for recorded_path, record in iter_file_records(payload):
            recorded = Path(recorded_path)
            same_path = (recorded.resolve() == path.resolve()) if recorded.is_absolute() else (recorded.as_posix().lstrip("./") == normalized)
            if same_path:
                matches.append((manifest_path, record))
    if not matches:
        raise RuntimeError(f"{relative_path} is not registered in a provenance manifest.")
    manifest_path, record = matches[0]
    expected = checksum_from_record(record)
    if not expected:
        raise RuntimeError(f"No SHA-256 is recorded for {relative_path} in {manifest_path}.")
    observed = sha256_file(path)
    if observed.lower() != expected.lower():
        raise RuntimeError(
            f"Checksum mismatch for {relative_path}: expected {expected}, observed {observed}."
        )
    return path


pd.DataFrame(
    {
        "manifest": [str(path.relative_to(ROOT)) for path, _ in MANIFESTS],
        "schema_version": [payload.get("schema_version", payload.get("version", "unspecified"))
                           for _, payload in MANIFESTS],
    }
)

## Figures 1 and S1

These provenance-locked manuscript snapshots connect the experimental design and analytical workflow to the validated units above. They are visual references rather than analytical inputs.

In [ ]:
for label, relative_path in {
    "Figure 1 — experimental design": "results/figures/main/figure_1.png",
    "Figure S1 — analytical workflow": "results/figures/supplement/figure_s1_workflow.png",
}.items():
    if (ROOT / relative_path).is_file():
        display(label, Image(filename=str(require_registered(relative_path))))
    else:
        print(f"{label}: optional manuscript-reference export is not present at {relative_path}.")

## Load the processed observations and eligible repertoires

A session repertoire contains all retained units produced by one caller in one recording session with one receiver. Sequence eligibility requires at least five phee sequences; call eligibility requires at least five individual phee calls.

In [ ]:
sequences = pd.read_csv(require_registered("data/processed/sequences.csv"))
calls = pd.read_csv(require_registered("data/processed/calls.csv"))
sequence_inventory = pd.read_csv(
    require_registered("data/cache/sequence_session_inventory.csv")
)
call_inventory = pd.read_csv(
    require_registered("data/cache/call_session_inventory.csv")
)
sequence_pairs_cached = pd.read_csv(
    require_registered("data/cache/sequence_session_pairs.csv")
)
call_pairs_cached = pd.read_csv(
    require_registered("data/cache/call_session_pairs.csv")
)

pair_columns = {"comparison_id", "pair_id", "stage", "context", "repertoire_a_id", "repertoire_b_id", "individual_a", "individual_b", "receiver_a", "receiver_b", "session_a", "session_b"}
required_columns = {
    "sequences": (sequences, {"sequence_id", "repertoire_id", "stage", "context"}),
    "calls": (calls, {"call_id", "repertoire_id", "stage", "context"}),
    "sequence inventory": (sequence_inventory, {"repertoire_id", "individual_id", "receiver_id", "pair_id", "session_number", "n_sequences", "stage", "context"}),
    "call inventory": (call_inventory, {"repertoire_id", "individual_id", "receiver_id", "pair_id", "session_number", "n_calls", "stage", "context"}),
    "sequence pair index": (sequence_pairs_cached, pair_columns),
    "call pair index": (call_pairs_cached, pair_columns),
}
for label, (table, columns) in required_columns.items():
    missing = sorted(columns.difference(table.columns))
    if missing:
        raise ValueError(f"{label} is missing canonical columns: {missing}")

if sequences["sequence_id"].duplicated().any():
    raise ValueError("data/processed/sequences.csv contains duplicate sequence_id values.")
if calls["call_id"].duplicated().any():
    raise ValueError("data/processed/calls.csv contains duplicate call_id values.")
if sequence_inventory["repertoire_id"].duplicated().any():
    raise ValueError("The sequence inventory contains duplicate repertoire_id values.")
if call_inventory["repertoire_id"].duplicated().any():
    raise ValueError("The call inventory contains duplicate repertoire_id values.")

{
    "processed_sequences": sequences.shape,
    "processed_calls": calls.shape,
    "eligible_sequence_repertoires": sequence_inventory.shape,
    "eligible_call_repertoires": call_inventory.shape,
}

## Manuscript count audit

The expected values below are fixed, non-computational validation targets transcribed from the manuscript. The observed values are calculated only from checksum-verified tables.

In [ ]:
def normalize_context(value: object) -> str:
    value = str(value).strip().lower().replace("_", "-")
    return "non-partner" if value in {"non-partner", "nonpartner", "stranger", "non-paired"} else value


import sys

src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from marmoset_convergence.sessions import build_session_pairs


def comparison_counts(table: pd.DataFrame) -> tuple[int, dict[str, int]]:
    return len(table), table.groupby("context")["comparison_id"].nunique().to_dict()


with (ROOT / "data/processed/manifest.json").open(encoding="utf-8") as handle:
    bonded_pairs = json.load(handle)["parameters"]["bonded_pairs"]
sequence_pairs_regenerated = build_session_pairs(sequence_inventory, bonded_pairs)
call_pairs_regenerated = build_session_pairs(call_inventory, bonded_pairs)


def require_exact_pair_index(cached: pd.DataFrame, regenerated: pd.DataFrame, label: str) -> None:
    if list(cached.columns) != list(regenerated.columns):
        raise AssertionError(f"{label} cached/regenerated column order differs.")
    try:
        pd.testing.assert_frame_equal(cached, regenerated, check_dtype=True, check_exact=True)
    except AssertionError as error:
        raise AssertionError(
            f"{label} registered pair index does not exactly equal the shared builder output."
        ) from error


require_exact_pair_index(sequence_pairs_cached, sequence_pairs_regenerated, "Sequence")
require_exact_pair_index(call_pairs_cached, call_pairs_regenerated, "Call")
sequence_pairs = sequence_pairs_cached
call_pairs = call_pairs_cached
n_seq_comparisons, seq_by_context = comparison_counts(sequence_pairs)
n_call_comparisons, call_by_context = comparison_counts(call_pairs)

observed = {
    "Analysed phee sequences": sequences["sequence_id"].nunique(),
    "Unique analysed phee calls": calls["call_id"].nunique(),
    "Eligible sequence repertoires": sequence_inventory["repertoire_id"].nunique(),
    "Sequences in eligible repertoires": int(sequence_inventory["n_sequences"].sum()),
    "Eligible call repertoires": call_inventory["repertoire_id"].nunique(),
    "Calls in eligible repertoires": int(call_inventory["n_calls"].sum()),
    "Unique sequence comparisons": n_seq_comparisons,
    "Partner sequence comparisons": seq_by_context.get("partner", 0),
    "Non-partner sequence comparisons": seq_by_context.get("non-partner", 0),
    "Unique call comparisons": n_call_comparisons,
    "Partner call comparisons": call_by_context.get("partner", 0),
    "Non-partner call comparisons": call_by_context.get("non-partner", 0),
}
expected = {
    "Analysed phee sequences": 1619,
    "Unique analysed phee calls": 3612,
    "Eligible sequence repertoires": 107,
    "Sequences in eligible repertoires": 1496,
    "Eligible call repertoires": 130,
    "Calls in eligible repertoires": 3527,
    "Unique sequence comparisons": 331,
    "Partner sequence comparisons": 62,
    "Non-partner sequence comparisons": 269,
    "Unique call comparisons": 431,
    "Partner call comparisons": 74,
    "Non-partner call comparisons": 357,
}

count_audit = pd.DataFrame(
    [{"quantity": key, "expected": expected[key], "observed": observed[key],
      "passes": expected[key] == observed[key]} for key in expected]
)
display(count_audit)
if not count_audit["passes"].all():
    failed = count_audit.loc[~count_audit["passes"], "quantity"].tolist()
    raise AssertionError(f"Manuscript count audit failed: {failed}")

## Design coverage

This table is descriptive. It makes missing stage-by-context cells visible before modelling. A missing cell is not filled or imputed here; the model’s partially pooled prediction and the common-support sensitivity analysis are documented in Notebook 05.

In [ ]:
coverage = (
    sequence_inventory.assign(context=sequence_inventory["context"].map(normalize_context))
    .groupby(["pair_id", "stage", "context"], observed=True)
    .agg(n_repertoires=("repertoire_id", "nunique"), n_sequences=("n_sequences", "sum"))
    .reset_index()
    .sort_values(["pair_id", "stage", "context"])
)
display(coverage)

## Interpretation

Passing this audit establishes the denominators used downstream. It does not test convergence. Notebook 02 validates the four call representations, Notebook 03 validates the four sequence representations, Notebook 04 validates the long-form model tables, and Notebook 05 summarizes the saved posterior draws.